# xrf-visualizer quickstart

Self-contained: this notebook **generates its own synthetic test cube**,
so no example data file is needed.

Prerequisites (one-time, see the README / `deploy/how_to_deploy_pip.md`):

```bash
uv venv xrf-visualizer-env            # create the environment
source xrf-visualizer-env/bin/activate   # Linux/macOS  (Windows: xrf-visualizer-env\Scripts\Activate.ps1)
uv pip install xrf-visualizer jupyterlab ipykernel
python -m ipykernel install --user --name xrf-visualizer --display-name "Python (xrf-visualizer)"
```

Then open this notebook with the **Python (xrf-visualizer)** kernel selected
(top-right) and run all cells.

In [ ]:
import xrf_visualizer

print("xrf-visualizer", xrf_visualizer.__version__)

## 1) Generate a synthetic test cube

Closed form (same recipe as `tests/make_fixture.py`): a background that
varies spatially, a bright rectangular region, and three Gaussian emission
peaks on the channel axis. The energy vector is an **arbitrary linear
calibration**, only to exercise the calibrated-axis UI.

In [ ]:
import numpy as np

H, W, C = 48, 64, 512
i, j = np.mgrid[0:H, 0:W]
c = np.arange(C)
base = 1.0 + 0.001 * ((i * W + j) % 7)
peaks = (
    1000.0 * np.exp(-(((c - 100.0) / 2.0) ** 2))
    + 500.0 * np.exp(-(((c - 250.0) / 3.0) ** 2))
    + 250.0 * np.exp(-(((c - 400.0) / 1.5) ** 2))
)
cube = (base[:, :, None] + peaks[None, None, :]).astype(np.float32)
cube[10:20, 30:40, :] *= 2.0  # bright region: visible in the map

energy = np.linspace(0.5, 40.5, C)  # arbitrary linear calibration for testing
np.savez_compressed("quickstart_cube.npz", img=cube, energy=energy)
print("saved quickstart_cube.npz", cube.shape, cube.dtype)

## 2) Load it and start the app

Left: 2-D map (sum over the selected channel interval). Right: spectrum of
the selected ROI. Try: drag a channel interval on the spectrum, draw an ROI
on the map, toggle log scale, select a spectral line.

In [ ]:
from xrf_visualizer.data import load_cube
from xrf_visualizer.ui import build_app

cube = load_cube("quickstart_cube.npz")  # XrfCube: array + energy axis
print(cube.path.name, cube.array.shape,
      "energy:", cube.energy[0], "-", cube.energy[-1], "keV")

app = build_app(cube)
app.run(jupyter_mode="inline", debug=False, jupyter_height=800)

## Your own data

- Point `load_cube(...)` at your own `.h5` / `.hdf5` / `.npz` / `.npy`
  file: key/dataset `img`, shape `(H, W, C)`, channels last; `.npz` may
  carry an `energy` vector (keV, strictly increasing).
- Integer count files (uint16/uint32, int16/int32) are accepted and
  converted to float64 at load.
- Or skip the file entirely: `app = build_app(None)` and use **Upload** or
  **Load from link** in the UI.

## Troubleshooting

- **App cell runs but nothing shows**: pick the kernel named
  *Python (xrf-visualizer)* (top-right); the environment must contain
  `xrf-visualizer`.
- **Port 8050 already in use** by another instance: close it, or run
  `app.run(jupyter_mode="inline", port=8051)`.
- **Outside a notebook**: `app.run()` starts a normal server on
  http://127.0.0.1:8050.